# nantrace on CUDA

Runs the nantrace test suite and the real bug benchmark on a GPU. In Colab choose Runtime, Change runtime type, GPU, then Run all. Please open an issue with the output, whether it passes or not.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version --format=csv
!pip -q install "nantrace[mcp] @ git+https://github.com/Optimus2007/nantrace" pytest lightning
!git clone -q https://github.com/Optimus2007/nantrace /content/nantrace

In [ ]:
import torch, nantrace
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('nantrace', nantrace.__version__)

In [ ]:
!cd /content/nantrace && python -m pytest -q tests

## The same checks on the GPU

Tracing on CUDA: an exp overflow in the forward pass, the torch.where gradient trap, and Guard replaying a bad step with dropout on the GPU.

In [ ]:
import torch
import nantrace

dev = 'cuda'
x = torch.tensor([[100.0, 1.0]], device=dev, requires_grad=True)
with nantrace.trace() as tr:
    w = torch.exp(x)
    p = w / w.sum(-1, keepdim=True)
print(tr.explain(p))

x = torch.tensor([0.0, 2.0], device=dev, requires_grad=True)
with nantrace.trace() as tr:
    torch.where(x > 0, torch.log(x), torch.zeros_like(x)).sum().backward()
print(tr.explain(x.grad))

In [ ]:
torch.manual_seed(0)
net = torch.nn.Sequential(torch.nn.Linear(16, 16), torch.nn.Dropout(0.3), torch.nn.Linear(16, 4)).to(dev)
opt = torch.optim.AdamW(net.parameters(), 1e-3)
guard = nantrace.Guard(net)

def loss_fn(x):
    w = net(x).exp()
    return (w / w.sum(-1, keepdim=True)).pow(2).sum(-1).mean()

try:
    for step in range(50):
        x = torch.randn(32, 16, device=dev)
        if step == 30:
            x[0] = 1e4
        guard.step(loss_fn, x)
        opt.step(); opt.zero_grad()
    print('no error: the replay did not reproduce, please report this')
except nantrace.NonFiniteError as e:
    print(e.report.text())

In [ ]:
!cd /content/nantrace && python benchmarks/overhead.py cuda

In [ ]:
!cd /content/nantrace/benchmarks/realworld && python score.py $(which nantrace) .